# 02 — Exploratory Data Analysis

**Dataset:** NHS Age-based Regression Dataset  
**Target:** `nhs_bmi__obese_class_1_rate` — Obesity Class 1 prevalence rate by single year of age  
**Goal:** Understand distributions, inter-feature relationships, and target behaviour before modelling.

In [21]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_seq_items', None)

warnings.filterwarnings("ignore")

# ── Styling ───────────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
PALETTE  = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B2"]
TARGET   = "nhs_bmi__obese_class_1_rate"

# ── Paths ─────────────────────────────────────────────────────────────────────
from pathlib import Path

ROOT = Path.cwd().parent   # go from Notebooks → main_folder
DATA = ROOT / "outputs_regression_dataset" / "regression_dataset.csv"
OUTDIR = ROOT / "outputs" / "eda"
OUTDIR.mkdir(parents=True, exist_ok=True)

## 1. Load & Prepare Data
Load the regression-ready CSV, keep only population-normalised **rate** columns, remove BMI sibling columns (data leakage), and apply variance + multicollinearity filters.

**Rationale:** Using rates (not raw counts) makes features comparable across age groups of different sizes. Variance filtering removes near-constant columns that add noise. Collinearity filtering (|r| > 0.97) prevents redundant features from destabilising linear models.

In [22]:
df_raw = pd.read_csv(DATA)
print(f"Dataset shape : {df_raw.shape[0]} rows × {df_raw.shape[1]} cols")
print(f"Age range     : {df_raw['age'].min()} – {df_raw['age'].max()}")
print(f"Missing values: {df_raw.isnull().sum().sum()}")

Dataset shape : 84 rows × 128 cols
Age range     : 16 – 99
Missing values: 0


In [23]:
df_raw.columns

Index(['age', 'nhs_abs_condition__all_conditions',
       'nhs_abs_condition__no_condition',
       'nhs_activity__did_not_meet_2014_physical_activity_guidelines',
       'nhs_activity__met_2014_physical_activity_guidelines',
       'nhs_alcohol__consumed_alcohol_12_or_more_months_ago',
       'nhs_alcohol__did_not_exceed_guideline',
       'nhs_alcohol__exceeded_guideline',
       'nhs_alcohol__never_consumed_alcohol',
       'nhs_alcohol__risk_level_not_known',
       'nhs_alcohol__time_since_last_consumed_alcohol_not_known',
       'nhs_bmi__normal_range', 'nhs_bmi__normal_range_adult_only',
       'nhs_bmi__obese_class_1', 'nhs_bmi__obese_class_2_adult_only',
       'nhs_bmi__obese_class_3_adult_only', 'nhs_bmi__overweight',
       'nhs_bmi__underweight_class_1', 'nhs_bmi__underweight_class_2',
       'nhs_bmi__underweight_class_3',
       'nhs_comorbidity_all_condition__certain_conditions_originating_in_the_perinatal_period',
       'nhs_comorbidity_all_condition__certain_infectio

In [24]:
# Keep only rate columns (population-normalised, age-comparable)
rate_cols = [c for c in df_raw.columns if c.endswith("_rate")]
df = df_raw[["age"] + rate_cols].copy()

# Remove BMI siblings → data leakage for our target
leak_patterns = ["nhs_bmi__"]
feature_cols = [
    c for c in rate_cols
    if c != TARGET and not any(p in c for p in leak_patterns)
]
X_raw = df[feature_cols].copy()
y     = df[TARGET].copy()

# Remove near-zero-variance features
vt = VarianceThreshold(threshold=1e-4)
vt.fit(X_raw)
feature_cols = [c for c, keep in zip(feature_cols, vt.get_support()) if keep]
X_raw = X_raw[feature_cols]

# Remove highly correlated pairs (|r| > 0.97)
corr_abs = X_raw.corr().abs()
upper    = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))
drop_set = {col for col in upper.columns if any(upper[col] > 0.97)}
feature_cols = [c for c in feature_cols if c not in drop_set]
X_raw = X_raw[feature_cols]

print(f"\nFeatures used for EDA : {len(feature_cols)}")

def short(col):
    """Human-readable column label."""
    return col.replace("nhs_", "").replace("_rate", "").replace("__", " | ")


Features used for EDA : 42


## 2. Target Variable Overview
We examine `nhs_bmi__obese_class_1_rate` through three lenses:
- **Histogram** — shape and spread
- **Age trend line** — non-linear peak in middle age
- **Q-Q plot** — departure from normality

> **Modelling implication:** The non-linear age curve and non-normal distribution justify tree-based models alongside linear ones.

In [32]:
df[TARGET]

0     0.183084
1     0.124913
2     0.037926
3     0.035576
4     0.107692
        ...   
79    0.000000
80    0.275510
81    0.000000
82    0.000000
83    0.000000
Name: nhs_bmi__obese_class_1_rate, Length: 84, dtype: float64

In [28]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Histogram
axes[0].hist(y, bins=20, color=PALETTE[0], edgecolor="white", linewidth=0.6)
axes[0].axvline(y.mean(), color="red", linestyle="--", linewidth=1.3, label=f"Mean={y.mean():.3f}")
axes[0].axvline(y.median(), color="orange", linestyle=":", linewidth=1.3, label=f"Median={y.median():.3f}")
axes[0].set_title("Distribution of Obesity Class 1 Rate")
axes[0].set_xlabel("Rate (proportion of age-group population)")
axes[0].set_ylabel("Count")
axes[0].legend(fontsize=8)

# Line plot vs age
axes[1].plot(df["age"], y, "o-", color=PALETTE[0], markersize=3, linewidth=1.4, alpha=0.85)
axes[1].fill_between(df["age"], y, alpha=0.12, color=PALETTE[0])
axes[1].set_title("Obesity Class 1 Rate by Age")
axes[1].set_xlabel("Age (years)")
axes[1].set_ylabel("Rate")

# QQ plot to check normality
from scipy import stats as scipy_stats
(osm, osr), (slope, intercept, r) = scipy_stats.probplot(y, dist="norm")
axes[2].plot(osm, osr, "o", markersize=4, color=PALETTE[1], alpha=0.8)
axes[2].plot(osm, slope * np.array(osm) + intercept, "r--", linewidth=1.2)
axes[2].set_title(f"Q-Q Plot (target)\nShapiro-Wilk p={scipy_stats.shapiro(y).pvalue:.4f}")
axes[2].set_xlabel("Theoretical Quantiles")
axes[2].set_ylabel("Sample Quantiles")

plt.suptitle("Target Variable: Obesity Class 1 Rate", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTDIR / "01_target_overview.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

print("Saved: 01_target_overview.png")

Saved: 01_target_overview.png


## 3. Correlation Heatmap
Pearson correlation between all retained features and the target (lower triangle only).

**What to look for:** Dark red/blue off-diagonal blocks signal multicollinearity → validates our choice of Ridge and Lasso regularisation.

In [38]:
heatmap_df = X_raw.copy()
heatmap_df["[TARGET]"] = y.values
corr_heat   = heatmap_df.corr()

corr_heat

,nhs_abs_condition__all_conditions_rate,nhs_activity__did_not_meet_2014_physical_activity_guidelines_rate,nhs_activity__met_2014_physical_activity_guidelines_rate,nhs_alcohol__consumed_alcohol_12_or_more_months_ago_rate,nhs_alcohol__did_not_exceed_guideline_rate,nhs_alcohol__exceeded_guideline_rate,nhs_alcohol__never_consumed_alcohol_rate,nhs_alcohol__time_since_last_consumed_alcohol_not_known_rate,nhs_comorbidity_all_condition__diseases_of_genito_urinary_system_rate,nhs_comorbidity_all_condition__diseases_of_the_blood_and_blood_forming_organs_rate,nhs_comorbidity_all_condition__diseases_of_the_circulatory_system_rate,nhs_comorbidity_all_condition__diseases_of_the_digestive_system_rate,nhs_comorbidity_all_condition__diseases_of_the_ear_and_mastoid_process_rate,nhs_comorbidity_all_condition__diseases_of_the_eye_and_adnexa_rate,nhs_comorbidity_all_condition__diseases_of_the_musculoskeletal_system_and_connective_tissue_rate,nhs_comorbidity_all_condition__diseases_of_the_nervous_system_rate,nhs_comorbidity_all_condition__diseases_of_the_respiratory_system_rate,nhs_comorbidity_all_condition__diseases_of_the_skin_and_subcutaneous_tissue_rate,nhs_comorbidity_all_condition__endocrine_nutritional_and_metabolic_disorders_rate,nhs_comorbidity_all_condition__injury_poisoning_and_certain_other_consequences_of_external_causes_rate,nhs_comorbidity_all_condition__mental_and_behavioural_disorders_rate,nhs_comorbidity_all_condition__neoplasms_rate,nhs_comorbidity_all_condition__symptoms_signs_and_conditions_not_elsewhere_classified_rate,nhs_seifa__decile_1_lowest_rate,nhs_seifa__decile_10_highest_rate,nhs_seifa__decile_2_rate,nhs_seifa__decile_3_rate,nhs_seifa__decile_4_rate,nhs_seifa__decile_5_rate,nhs_seifa__decile_6_rate,nhs_seifa__decile_7_rate,nhs_seifa__decile_8_rate,nhs_seifa__decile_9_rate,nhs_smoking__current_daily_smoker_rate,nhs_smoking__current_non_smoker_rate,nhs_smoking__current_smoker_other_rate,nhs_states__new_south_wales_rate,nhs_states__queensland_rate,nhs_states__south_australia_rate,nhs_states__tasmania_rate,nhs_states__victoria_rate,nhs_states__western_australia_rate,[TARGET]
nhs_abs_condition__all_conditions_rate,1.000000,0.056022,-0.121399,0.361721,-0.180218,-0.153625,-0.160010,0.104337,0.299929,0.017317,0.787633,0.445924,0.596077,0.899040,0.725729,-0.076010,-0.096778,-0.110767,0.655278,-0.057482,-0.342489,0.697834,0.468804,-0.130405,-0.262050,0.055061,-0.020972,-0.031282,-0.026556,0.047736,-0.129311,-0.236016,0.015581,-0.100550,0.219804,-0.445714,-0.147227,-0.243877,0.084857,0.093392,-0.163790,-0.155991,0.186622
nhs_activity__did_not_meet_2014_physical_activity_guidelines_rate,0.056022,1.000000,-0.597048,-0.324369,-0.174360,0.011919,0.263194,-0.051939,-0.116171,0.078563,0.089258,-0.163445,-0.154946,0.137616,0.183330,-0.043151,-0.008349,-0.123800,0.042977,0.124327,-0.035944,-0.153090,-0.189760,-0.397443,-0.010733,-0.136514,-0.006212,0.047675,0.090383,0.010067,-0.040244,0.006544,-0.119465,0.097725,0.037849,-0.154802,-0.305662,-0.216344,-0.116768,0.121665,-0.080895,-0.097489,-0.102832
nhs_activity__met_2014_physical_activity_guidelines_rate,-0.121399,-0.597048,1.000000,0.186217,0.510595,0.307540,-0.189354,0.139172,0.154316,-0.002187,-0.139917,0.433962,-0.224223,-0.118292,-0.147934,0.154597,0.149426,0.351004,0.093767,0.430745,0.138010,-0.008923,-0.133886,0.215055,0.237870,0.132787,0.217461,0.260010,-0.108171,0.256533,0.104035,0.184001,0.439081,0.069984,0.006764,0.245377,0.428759,0.444263,-0.045965,0.033916,0.429540,0.311361,0.342856
nhs_alcohol__consumed_alcohol_12_or_more_months_ago_rate,0.361721,-0.324369,0.186217,1.000000,0.015745,-0.314981,-0.120299,0.175777,0.230728,0.179303,0.405672,0.302034,0.382288,0.261082,0.322851,0.000949,0.038434,0.037385,0.302429,0.024415,-0.064446,0.515081,0.622710,-0.058916,-0.008460,0.048145,0.043121,0.177581,-0.199041,0.121273,0.071451,-0.028848,0.317505,-0.194287,0.308442,-0.241050,0.123508,0.004524,0.419078,-0.058855,-0.008959,0.243150,0.175206
nhs_alcohol__did_not_exceed_guideline_rate,-0

In [33]:
heatmap_df = X_raw.copy()
heatmap_df["[TARGET]"] = y.values
corr_heat   = heatmap_df.corr()
short_lbls  = [short(c) for c in corr_heat.columns]

fig, ax = plt.subplots(figsize=(15, 12))
mask = np.triu(np.ones_like(corr_heat, dtype=bool))
sns.heatmap(
    corr_heat, mask=mask, annot=True, fmt=".2f",
    cmap="coolwarm", center=0, linewidths=0.25, linecolor="white",
    ax=ax, xticklabels=short_lbls, yticklabels=short_lbls,
    annot_kws={"size": 6.5}, vmin=-1, vmax=1,
)
ax.set_title("Pearson Correlation Heatmap – Retained Features + Target",
             fontsize=13, fontweight="bold", pad=12)
plt.xticks(rotation=45, ha="right", fontsize=7.5)
plt.yticks(fontsize=7.5)
plt.tight_layout()
plt.savefig(OUTDIR / "02_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: 02_correlation_heatmap.png")

Saved: 02_correlation_heatmap.png


## 4. Top-5 Features vs Target
Scatter plots with OLS regression line and 95% confidence band for the five features most correlated with the obesity rate.

> Features with |r| > 0.30 and low p-values are strong predictors; we also check whether the relationship is approximately linear.

In [39]:
corr_with_target = X_raw.corrwith(y).abs().sort_values(ascending=False)
top5 = corr_with_target.head(5).index.tolist()

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, col in zip(axes, top5):
    x_vals = X_raw[col].values
    # scatter
    ax.scatter(x_vals, y.values, alpha=0.65, s=22, color=PALETTE[1], edgecolor="white", linewidth=0.3)
    # OLS line + 95 % CI band
    slope, intercept, r_val, p_val, stderr = scipy_stats.linregress(x_vals, y.values)
    xl  = np.linspace(x_vals.min(), x_vals.max(), 200)
    yl  = slope * xl + intercept
    n   = len(x_vals)
    se  = stderr * np.sqrt(1/n + (xl - x_vals.mean())**2 / ((n-1)*x_vals.var()))
    t95 = scipy_stats.t.ppf(0.975, df=n-2)
    ax.plot(xl, yl, "r-", linewidth=1.3)
    ax.fill_between(xl, yl - t95*se, yl + t95*se, alpha=0.15, color="red")
    ax.set_xlabel(short(col), fontsize=7.5)
    ax.set_ylabel("Obesity Rate" if ax is axes[0] else "")
    ax.set_title(f"|r| = {corr_with_target[col]:.3f}\np = {p_val:.3f}", fontsize=8.5)

plt.suptitle("Top-5 Correlated Features vs Obesity Class 1 Rate",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUTDIR / "03_top5_scatter.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: 03_top5_scatter.png")

Saved: 03_top5_scatter.png


## 5. Feature Distributions by Domain
Boxplots grouped by NHS data domain (activity, alcohol, comorbidity, seifa, smoking, states). Outlier-heavy distributions may unduly influence linear models.

In [40]:
def domain(col):
    parts = col.replace("nhs_", "").split("__")
    return parts[0] if len(parts) > 1 else "other"

domain_map = {c: domain(c) for c in feature_cols}
domains    = sorted(set(domain_map.values()))

fig, axes = plt.subplots(len(domains), 1,
                          figsize=(14, 3.5 * len(domains)),
                          constrained_layout=True)
if len(domains) == 1:
    axes = [axes]

for ax, dom in zip(axes, domains):
    cols = [c for c in feature_cols if domain_map[c] == dom]
    data = [X_raw[c].dropna().values for c in cols]
    bp   = ax.boxplot(data, patch_artist=True, vert=True,
                      medianprops=dict(color="red", linewidth=1.5))
    for patch, colour in zip(bp["boxes"], plt.cm.tab20.colors):
        patch.set_facecolor(colour)
        patch.set_alpha(0.7)
    ax.set_xticks(range(1, len(cols) + 1))
    ax.set_xticklabels([short(c) for c in cols], rotation=40, ha="right", fontsize=8)
    ax.set_title(f"Domain: {dom}", fontsize=10, fontweight="bold")
    ax.set_ylabel("Rate")

plt.suptitle("Feature Distributions by Domain (Boxplots)", fontsize=13,
             fontweight="bold")
plt.savefig(OUTDIR / "04_feature_boxplots.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: 04_feature_boxplots.png")

Saved: 04_feature_boxplots.png


## 6. Age-Stratified Heatmap
Mean rate of top-5 features + target aggregated into 10-year age bands. Reveals how health patterns shift across the lifespan.

In [41]:
# Bin ages into 10-year bands
age_bins   = list(range(16, 100, 10)) + [100]
age_labels = [f"{a}–{a+9}" for a in age_bins[:-1]]
df2 = df.copy()
df2["age_band"] = pd.cut(df2["age"], bins=age_bins, labels=age_labels, right=False)

# Pick a representative subset of features
highlight_cols = top5 + [TARGET]
agg = df2.groupby("age_band", observed=True)[highlight_cols].mean()

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    agg.T, annot=True, fmt=".3f", cmap="YlOrRd",
    linewidths=0.4, linecolor="white", ax=ax,
    xticklabels=age_labels,
    yticklabels=[short(c) for c in highlight_cols],
    annot_kws={"size": 8},
)
ax.set_title("Mean Rate by Age Band – Top Features + Target",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Age Band")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig(OUTDIR / "05_age_band_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: 05_age_band_heatmap.png")

Saved: 05_age_band_heatmap.png


## 7. EDA Summary Table
Descriptive statistics and correlation-with-target for every retained feature, sorted by |correlation|. Saved to `outputs/eda/eda_summary.csv`.

In [42]:
desc = X_raw[feature_cols].describe().T
desc["corr_with_target"] = X_raw.corrwith(y)
desc["abs_corr"]         = desc["corr_with_target"].abs()
desc = desc.sort_values("abs_corr", ascending=False)
desc.to_csv(OUTDIR / "eda_summary.csv")
print("Saved: eda_summary.csv")

print("\n── Top-10 features by |correlation with target| ──")
print(desc[["mean", "std", "corr_with_target"]].head(10).to_string())
print(f"\nAll EDA outputs saved to: {OUTDIR}")

Saved: eda_summary.csv

── Top-10 features by |correlation with target| ──
                                                                                      mean       std  corr_with_target
nhs_comorbidity_all_condition__diseases_of_the_digestive_system_rate              0.085317  0.063153          0.553014
nhs_alcohol__did_not_exceed_guideline_rate                                        0.466085  0.123753          0.489654
nhs_states__queensland_rate                                                       0.181366  0.078800          0.403982
nhs_seifa__decile_4_rate                                                          0.106905  0.055785          0.400014
nhs_comorbidity_all_condition__diseases_of_the_skin_and_subcutaneous_tissue_rate  0.041253  0.027524          0.392375
nhs_states__new_south_wales_rate                                                  0.294416  0.098108          0.379280
nhs_states__victoria_rate                                                         0.229802  